In [16]:
%pip install beautifulsoup4 requests
%pip install aiohttp

from bs4 import BeautifulSoup

import pandas as pd

import _asyncio
import aiohttp
import time


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


aiohttp, asenkron HTTP istemcisi ve sunucusu sağlayan bir kütüphanedir. Web isteklerini asenkron bir şekilde yapmanıza olanak tanır ve genellikle hızlı ve verimli veri çekme işlemleri için kullanılır.

In [17]:
# https://www.hukuksorucevap.com.tr/sorucevap/?sayfa=1 adresindeki soru ve cevapları çekeceğiz. Toplamda 36 sayfa var.

In [18]:
def temizle(metin):
    return metin.replace("\n", " ").replace("\t", " ").replace("\r", " ").strip()
    while "  " in metin:
        metin = metin.replace("  ", " ")
    return metin.strip()

In [19]:
tum_sorular = [] # her defasında sıfırlamamızı saglıyor

async def sayfadaki_sorulari_cek(session, i):
    try:
        async with session.get('https://www.hukuksorucevap.com.tr/sorucevap/?sayfa=' + str(i)) as response:
         response.encoding = 'utf-8'    
        
        soup = BeautifulSoup(await response.text, 'html.parser')
        soru = soup.find('div', {'id': 'Soru'})
        # len(soru)
        sorular = soru.findAll('li')
        len(sorular)
        for soru_li in sorular:
            soru_dict = {}
            soru_metni = soru_li.find('div', {'id': 'Soru'}).text
            soru_dict['soru'] = temizle(soru_metni)

            cevap_metni = soru_li.find('div', {'id': 'Cevap'}).find('div', {'class': 'DoktorCevabi'}).text
            soru_dict['cevap'] = temizle(cevap_metni)

            cevaplayan_unvan = soru_li.find('div', {'id': 'Cevap'}).find('div', {'class': 'DoktorUnvani'}).text
            soru_dict['cevaplayan_unvan'] = temizle(cevaplayan_unvan)
    
            tum_sorular.append((soru_dict))


    except Exception as e:
        print(e)

In [24]:
import asyncio

start_time = time.time()
async with aiohttp.ClientSession() as session:
    tasks = []
    for i in range(1, 37):
        tasks.append(sayfadaki_sorulari_cek(session, i))
    await asyncio.gather(*tasks)
end_time = time.time()

print(f"Time taken to get the details of profiles: {end_time - start_time} seconds")

object method can't be used in 'await' expression
object method can't be used in 'await' expression
object method can't be used in 'await' expression
object method can't be used in 'await' expression
object method can't be used in 'await' expression
object method can't be used in 'await' expression
Cannot connect to host www.hukuksorucevap.com.tr:443 ssl:default [Connect call failed ('109.232.220.133', 443)]
Cannot connect to host www.hukuksorucevap.com.tr:443 ssl:default [Connect call failed ('109.232.220.133', 443)]
Cannot connect to host www.hukuksorucevap.com.tr:443 ssl:default [Connect call failed ('109.232.220.133', 443)]
Cannot connect to host www.hukuksorucevap.com.tr:443 ssl:default [Connect call failed ('109.232.220.133', 443)]
Cannot connect to host www.hukuksorucevap.com.tr:443 ssl:default [Connect call failed ('109.232.220.133', 443)]
Cannot connect to host www.hukuksorucevap.com.tr:443 ssl:default [Connect call failed ('109.232.220.133', 443)]
Cannot connect to host www.h

In [25]:
# convert tum_sorular to a DataFrame
df = pd.DataFrame(tum_sorular)
df

""


In [26]:
df.to_csv('hukuksorucevap.csv', index=False, encoding='utf-8')

In [27]:
df_csv = pd.read_csv('hukuksorucevap.csv')
df_csv

EmptyDataError: No columns to parse from file